# Consultas y validación de tablas Gold

Notebook de solo lectura para Azure Databricks.
Cada tabla tiene una celda independiente de consulta.
Los conteos esperados corresponden a la carga inicial aceptada de M9;
deben revisarse cuando se incorporen nuevas cargas.
Las comprobaciones complementan la validación del notebook de construcción.

## Consulta de `dim_product`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_sales.dim_product
ORDER BY product_key
LIMIT 100;

## Consulta de `dim_customer`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_sales.dim_customer
ORDER BY customer_key
LIMIT 100;

## Consulta de `dim_date`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_sales.dim_date
ORDER BY date_key
LIMIT 100;

## Consulta de `dim_order_status`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_sales.dim_order_status
ORDER BY order_status_key
LIMIT 100;

## Consulta de `fact_sales`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_sales.fact_sales
ORDER BY order_id, line_number
LIMIT 100;

## Consulta de `ref_exchange_rates`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_reference.ref_exchange_rates
ORDER BY series_code, observation_date
LIMIT 100;

## Consulta de `product_catalog_mapping`

In [ ]:
%sql
SELECT *
FROM dbw_arlp_dev_diego01.gold_reference.product_catalog_mapping
ORDER BY product_key, source_system, external_product_id
LIMIT 100;

## Conteos de la carga inicial

La celda falla si algún conteo difiere del valor aceptado.

In [ ]:
targets = {'dim_product': ('dbw_arlp_dev_diego01.gold_sales.dim_product', 61), 'dim_customer': ('dbw_arlp_dev_diego01.gold_sales.dim_customer', 500), 'dim_date': ('dbw_arlp_dev_diego01.gold_sales.dim_date', 730), 'dim_order_status': ('dbw_arlp_dev_diego01.gold_sales.dim_order_status', 3), 'fact_sales': ('dbw_arlp_dev_diego01.gold_sales.fact_sales', 5862), 'ref_exchange_rates': ('dbw_arlp_dev_diego01.gold_reference.ref_exchange_rates', 195), 'product_catalog_mapping': ('dbw_arlp_dev_diego01.gold_reference.product_catalog_mapping', 6)}

results = []
for name, (full_name, expected) in targets.items():
    actual = spark.table(full_name).count()
    results.append((name, expected, actual, actual == expected))

display(spark.createDataFrame(
    results, ["table_name", "expected_rows", "actual_rows", "passed"]
))
assert all(row[3] for row in results), "Conteos Gold diferentes de lo esperado."
print("PASS: conteos de las siete tablas.")

## Claves dimensionales y grano de ventas

Comprueba claves no nulas y únicas en dimensiones y tasas.
En ventas comprueba el grano: una fila por pedido y número de línea.

In [ ]:
from pyspark.sql import functions as F

key_checks = {
    "dim_product": ["product_key"],
    "dim_customer": ["customer_key"],
    "dim_date": ["date_key"],
    "dim_order_status": ["order_status_key"],
    "fact_sales": ["order_id", "line_number"],
    "ref_exchange_rates": ["series_code", "observation_date"],
}

for name, keys in key_checks.items():
    frame = spark.table(targets[name][0])
    null_key = F.lit(False)
    for key in keys:
        null_key = null_key | F.col(key).isNull()

    assert not frame.filter(null_key).limit(1).count(), (
        f"Clave nula en {name}: {keys}"
    )
    assert not (
        frame.groupBy(*keys).count()
        .filter(F.col("count") > 1).limit(1).count()
    ), f"Clave duplicada en {name}: {keys}"
    print(f"PASS: {name}, claves {keys}")

## Integridad referencial de ventas

Comprueba que cada venta tenga correspondencia en las cuatro dimensiones.

In [ ]:
sales = spark.table(targets["fact_sales"][0])

for dimension, key in (
    ("dim_product", "product_key"),
    ("dim_customer", "customer_key"),
    ("dim_date", "date_key"),
    ("dim_order_status", "order_status_key"),
):
    dimension_keys = spark.table(targets[dimension][0]).select(key)
    missing = sales.join(dimension_keys, key, "left_anti")
    assert not missing.limit(1).count(), (
        f"Ventas sin correspondencia en {dimension}"
    )
    print(f"PASS: relación fact_sales → {dimension}")

## Consistencia de importes

Comprueba cantidad positiva, importes PEN y neto por diferencia.
En USD comprueba la disponibilidad de conversión y el neto por diferencia.
Esta comprobación no recalcula la selección de tasas ni toda la conversión.

In [ ]:
required = [
    "quantity", "unit_price_pen", "gross_amount_pen",
    "discount_amount_pen", "net_amount_pen", "conversion_available",
]
invalid = F.lit(False)
for column in required:
    invalid = invalid | F.col(column).isNull()

invalid = (
    invalid
    | (F.col("quantity") <= 0)
    | (F.col("unit_price_pen") < 0)
    | (F.col("discount_amount_pen") < 0)
    | (F.col("discount_amount_pen") > F.col("gross_amount_pen"))
    | (F.col("gross_amount_pen")
       != F.col("quantity") * F.col("unit_price_pen"))
    | (F.col("net_amount_pen")
       != F.col("gross_amount_pen") - F.col("discount_amount_pen"))
)
assert not sales.filter(invalid).limit(1).count(), "Importes PEN inválidos."

usd_columns = [
    "gross_amount_usd", "discount_amount_usd", "net_amount_usd"
]
missing_usd = F.lit(False)
present_usd = F.lit(False)
for column in usd_columns:
    missing_usd = missing_usd | F.col(column).isNull()
    present_usd = present_usd | F.col(column).isNotNull()

invalid_usd = (
    (F.col("conversion_available") & (
        missing_usd
        | F.col("rate_pen_per_usd").isNull()
        | (F.col("rate_pen_per_usd") <= 0)
        | (F.col("net_amount_usd")
           != F.col("gross_amount_usd") - F.col("discount_amount_usd"))
    ))
    | (~F.col("conversion_available") & present_usd)
)
assert not sales.filter(invalid_usd).limit(1).count(), "Importes USD inválidos."
print("PASS: consistencia básica de importes PEN y USD.")

## Ventas por categoría

Incluye únicamente estados con `counts_as_sale = true`.
Los importes se presentan en PEN.

In [ ]:
%sql
SELECT
    p.category_name,
    COUNT(*) AS sales_lines,
    COUNT(DISTINCT f.order_id) AS orders,
    SUM(f.quantity) AS units,
    SUM(f.gross_amount_pen) AS gross_sales_pen,
    SUM(f.discount_amount_pen) AS discounts_pen,
    SUM(f.net_amount_pen) AS net_sales_pen
FROM dbw_arlp_dev_diego01.gold_sales.fact_sales AS f
JOIN dbw_arlp_dev_diego01.gold_sales.dim_product AS p
    ON f.product_key = p.product_key
JOIN dbw_arlp_dev_diego01.gold_sales.dim_order_status AS s
    ON f.order_status_key = s.order_status_key
WHERE s.counts_as_sale = true
GROUP BY p.category_name
ORDER BY net_sales_pen DESC;